<a href="https://colab.research.google.com/github/thepipo93/etl-dashboard-ipalmera/blob/main/Avance_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Statistical and Predictive Analysis in AdTech: Optimizing Cost Per Acquisition (CPA) through Machine Learning**

Authors: "Juan Sebastian Hoyos Espinosa, Koraima Torres Díaz, Alexandra Libreros Castillo, Christian Felipe Trujillo Franco"

In [ ]:
# ==========================================
# 1. IMPORTAMOS LIBRERÍAS
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# 1. Introduction and Problem Statement

## Problem Definition

This project is developed within the Digital Advertising ecosystem, where agencies and Media Traders manage massive investments across multiple channels (e.g., Google Ads, Facebook, LinkedIn) (Smith & Chaffey, 2023). Currently, key performance indicators (KPIs) such as the Cost Per Acquisition (CPA) exhibit high daily volatility influenced by the diversity of creative formats and investment levels.

The critical problem is operational uncertainty. Analysts observe performance variations, but these lack a supporting statistical framework (Provost & Fawcett, 2013). Without robust validation, there is a risk of reacting to random market fluctuations—erroneously scaling or pausing campaigns—rather than identifying true competitive advantages. The central challenge consists of distinguishing whether a platform's efficiency is a random event or a constant structural superiority.

##**Research Question:**
Are there statistically significant differences in CPA among different platforms and creative formats that operationally justify a reallocation of the advertising budget?

## 1.2 Objectives

### General Objective

To statistically evaluate the performance of digital campaigns through the analysis of their operational costs, aiming to optimize investment and ground strategic scaling decisions using mathematical evidence and predictive modeling.

### Specific Objectives

- To characterize the distributional behavior of CPA and ad spend, identifying biases and outliers to establish a reliable performance baseline.

- To estimate 95% confidence intervals for the population CPA based on creative format, providing safe reference ranges for financial planning.

- To compare performance across platforms using parametric and non-parametric hypothesis testing, identifying those that structurally minimize acquisition costs compared to their competitors.

- To develop and evaluate predictive Machine Learning models to classify the probability of a new advertising campaign achieving cost-efficiency, allowing for proactive budget optimization and allocation.

## 1.3 Methodology

To achieve these objectives, the analysis was structured into four progressive phases:

- **Phase 1 — Exploratory Data Analysis (EDA):** A multivariate statistical characterization was executed to evaluate dataset integrity (Tukey, 1977). This initial phase focused on calculating statistical moments and generating density histograms and dispersion boxplots, allowing for the identification of extreme skewness and kurtosis patterns. This diagnosis was fundamental to determining the viability of probability models and detecting outliers that could distort central tendency metrics.

- **Phase 2 — Interval Estimation:** To establish operational thresholds by creative format, a 95% confidence interval design was implemented. A parametric approach (based on the Central Limit Theorem) was contrasted with a robust Bootstrap approach over the median. This methodological duality mitigated the impact of outliers and provided much more stable and realistic expected cost references for the agency's daily operations.

- **Phase 3 — Hypothesis Testing:** The third phase comprised formal hypothesis testing. First, group distributions were verified using the Lilliefors test to determine the appropriate comparison technique. After applying a Pareto Analysis to isolate the platforms concentrating 80% of the ad spend, the non-parametric Kruskal-Wallis test was selected to evaluate the equality of efficiency ranks. Finally, upon detecting significant differences, Dunn's test with Bonferroni correction was executed, ensuring strict control of Type I error in multiple comparisons.

- **Phase 4 — Predictive Modeling (Machine Learning):** A classification pipeline was built to predict whether a campaign will be "Cost-Efficient" (Target Variable). Feature engineering (One-Hot Encoding) was applied to categorical variables such as platform and format. Subsequently, metric data were standardized using StandardScaler. Training was conducted by splitting the dataset into an 80% training set and a 20% testing set. The performance of a baseline model (Logistic Regression) was compared against an advanced model (Support Vector Machine with RBF Kernel), evaluating their efficacy through the Recall metric and the Confusion Matrix to minimize the operational risk of false positives.

In [ ]:
# ==========================================
# 2. CARGA Y PREPARACIÓN DE DATOS
# ==========================================
#Importamos Doc
from google.colab import files
uploaded = files.upload()
# Tras Importar
df_raw = pd.read_csv('tech_advertising_campaigns_dataset.csv')
# Filtramos las columnas
df = df_raw[['platform', 'creative_format', 'ad_spend', 'conversions', 'CPA']].copy()

In [ ]:
# ==========================================
# 3. ANÁLISIS EXPLORATORIO (Estadística Descriptiva)
# ==========================================
# Cargar datos
df_raw = pd.read_csv('tech_advertising_campaigns_dataset.csv')
df = df_raw[['platform', 'creative_format', 'ad_spend', 'conversions', 'CPA']].copy()

# Seleccionar solo las columnas numéricas
df_numeric = df.select_dtypes(include=[np.number])

# Crear el resumen estadístico (Media, Mediana, Min, Max, SD)
resumen_stats = df_numeric.describe().T
resumen_stats['mediana'] = df_numeric.median()

# Calcular Asimetría y Curtosis
resumen_stats['asimetria'] = df_numeric.apply(lambda x: skew(x.dropna()))
resumen_stats['curtosis'] = df_numeric.apply(lambda x: kurtosis(x.dropna(), fisher=True))

# Reorganizar columnas para que quede como en tu Rmd
resumen_stats = resumen_stats[['mean', 'std', 'mediana', 'min', 'max', 'asimetria', 'curtosis']]
resumen_stats.columns = ['Media', 'Desv. Est.', 'Mediana', 'Min', 'Max', 'Asimetría', 'Curtosis']

print("--- ANÁLISIS DESCRIPTIVO DE LAS VARIABLES MÉTRICAS ---")
display(resumen_stats.round(2))

# ==========================================
# 3.1: VISUALIZACIONES (EDA)
# ==========================================
# Configurar el estilo general de los gráficos
sns.set_theme(style="whitegrid")

# Crear un lienzo (Figure) con 2 filas y 2 columnas para gráficos
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.tight_layout(pad=5.0) # Espacio entre gráficos

# --- Gráfico 1 (Arriba Izquierda): Histograma + Densidad del CPA ---
sns.histplot(data=df, x='CPA', kde=True, bins=50, color='steelblue', ax=axes[0, 0])
axes[0, 0].set_title('Distribución del CPA (Costo por Adquisición)', fontweight='bold')
axes[0, 0].set_xlabel('CPA ($)')
axes[0, 0].set_ylabel('Densidad')

# --- Gráfico 2 (Arriba Derecha): Barras de Plataforma ---
# Ordenar por frecuencia
orden_plataformas = df['platform'].value_counts().index
sns.countplot(data=df, x='platform', order=orden_plataformas, palette='viridis', ax=axes[0, 1])
axes[0, 1].set_title('Distribución de Campañas por Plataforma', fontweight='bold')
axes[0, 1].set_xlabel('Plataforma')
axes[0, 1].set_ylabel('Cantidad')
axes[0, 1].tick_params(axis='x', rotation=30)

# --- Gráfico 3 (Abajo Izquierda): Boxplot de CPA por Plataforma ---
sns.boxplot(data=df, x='platform', y='CPA', palette='Set2', ax=axes[1, 0])
axes[1, 0].set_title('Eficiencia de Costo (CPA) por Plataforma', fontweight='bold')
axes[1, 0].set_xlabel('Plataforma')
axes[1, 0].set_ylabel('CPA ($)')
axes[1, 0].tick_params(axis='x', rotation=30)

# --- Gráfico 4 (Abajo Derecha): Dispersión Inversión vs Conversiones ---
# sns.regplot hace el scatter plot y automáticamente dibuja la línea de tendencia (lm)
sns.regplot(data=df, x='ad_spend', y='conversions', scatter_kws={'alpha':0.3, 'color':'darkgreen'}, line_kws={'color':'red'}, ax=axes[1, 1])
axes[1, 1].set_title('Relación Inversión vs. Conversiones', fontweight='bold')
axes[1, 1].set_xlabel('Inversión ($)')
axes[1, 1].set_ylabel('Conversiones')

plt.show()

2.1 Exploratory Data Analysis (EDA) Results and Conclusions

Based on the extraction of descriptive statistics and multivariate visual inspection, fundamental structural patterns in the behavior of advertising campaigns were identified:

1. Positive Skewness and Extreme Outliers (CPA Distribution):
The descriptive analysis reveals that all business metrics exhibit non-normal distributions with heavy right tails. The Cost Per Acquisition (CPA) shows a mean of $190.90, which is severely distorted by outliers (a maximum of $6,379.94), as evidenced by a skewness coefficient of 5.74 and an extreme kurtosis of 61.46.

Business Insight: Most campaigns operate within an efficient range (median of $99.46), but a small subset of "toxic" campaigns is severely draining the budget.

Machine Learning Implication: This high dispersion makes the use of scaling techniques (e.g., StandardScaler) mandatory before training the model, as algorithms like SVM are highly sensitive to extreme magnitudes and skewness.

2. Operational Concentration by Platform:
The campaign distribution chart demonstrates that the agency's operational efforts are not homogeneous. Facebook and Google Ads overwhelmingly dominate the volume of active campaigns, followed by LinkedIn.

Business Insight: This concentration justifies the subsequent application of a Pareto Analysis to focus hypothesis testing (Kruskal-Wallis) solely on the top 3 platforms, which represent the agency's true financial risk.

3. Efficiency Volatility (Platform Boxplots):
Observing the CPA dispersion by platform makes it evident that efficiency boundaries vary drastically. Certain platforms possess compact medians but a massive volume of upper outliers (data points beyond the boxplot whiskers).

Business Insight: This corroborates the central problem statement: Media Traders face high daily uncertainty. Monitoring the average cost is insufficient; the standard deviation of these costs is critically high.

4. Law of Diminishing Returns (Ad Spend vs. Conversions):
The scatterplot highlights a general positive correlation: higher investment (ad_spend) generally yields more conversions. However, a phenomenon of heteroscedasticity is observed (data dispersion widens as investment grows).

Business Insight: Scaling a campaign's budget does not guarantee proportionally higher conversions. High-budget campaigns exhibit highly erratic behavior.

Machine Learning Implication: The predictive model will be vital precisely to identify which campaigns maintain a stable conversion rate as they scale, avoiding budget waste in the plot's "noise zone."

In [ ]:
# ==========================================
# 4. INGENIERÍA DE CARACTERÍSTICAS (Feature Engineering)
# ==========================================
# Creamos la variable objetivo 'Exito'
df['Exito'] = np.where(df['CPA'] < 100, 1, 0)
# Ahora, la máquina no lee texto como "Facebook" o "Video".
# Convertimos estas variables a números con One-Hot Encoding.
df_ml = pd.get_dummies(df, columns=['platform', 'creative_format'], drop_first=True)

# Separamos la variable que queremos predecir (Y) de los datos de la campaña (X)
# Quitamos la columna CPA original porque sería hacer trampa (si la máquina sabe el CPA exacto, no necesita predecir).
X = df_ml.drop(['CPA', 'Exito'], axis=1)
y = df_ml['Exito']

In [ ]:
# ==========================================
# 5. SPLIT Y ESCALADO (StandardScaler)
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ==========================================
# 6. ENTRENAMIENTO SVM (Fase 4 de tu Metodología)
# ==========================================
print("Entrenando Máquina de Vectores de Soporte (Kernel RBF)...")
modelo_svm = SVC(kernel='rbf', C=1.0, random_state=42)
modelo_svm.fit(X_train_scaled, y_train)

predicciones = modelo_svm.predict(X_test_scaled)

In [ ]:
# ==========================================
# 7. EVALUACIÓN DE RESULTADOS
# ==========================================
print("\n--- REPORTE DEL MODELO ---")
print(classification_report(y_test, predicciones, target_names=['Fracaso (CPA Alto)', 'Exito (CPA Bajo)']))

plt.figure(figsize=(6,5))
sns.heatmap(confusion_matrix(y_test, predicciones), annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicción del Modelo')
plt.ylabel('Realidad')
plt.title('Matriz de Confusión - Detección de Campañas Exitosas')
plt.show()